In [1]:
import sys
print(sys.executable)

c:\Users\User\Documents\JR\loan-risk-prediction\loanriskenv\Scripts\python.exe


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import matthews_corrcoef

from scipy.stats import chi2_contingency, pointbiserialr
from scipy.stats import boxcox
from scipy import stats

import miceforest as mf


## **<i><strong>1. Data Loading**</i></strong>

In [3]:
loan_db = pd.read_csv('../assets/loan.csv', keep_default_na=False, na_values=['', 'NA', 'N/A'])
loan_db.head()

,loanId,anon_ssn,payFrequency,apr,applicationDate,originated,originatedDate,nPaidOff,approved,isFunded,loanStatus,loanAmount,originallyScheduledPaymentAmount,state,leadType,leadCost,fpStatus,clarityFraudId,hasCF
0,LL-I-07399092,beff4989be82aab4a5b47679216942fd,B,360.0,2016-02-23T17:29:01.940000,False,NaN,0.0,False,0,Withdrawn Application,500.0,978.27,IL,bvMandatory,6,NaN,5669ef78e4b0c9d3936440e6,1
1,LL-I-06644937,464f5d9ae4fa09ece4048d949191865c,B,199.0,2016-01-19T22:07:36.778000,True,2016-01-20T15:49:18.846000,0.0,True,1,Paid Off Loan,3000.0,6395.19,CA,prescreen,0,Checked,569eb3a3e4b096699f685d64,1
2,LL-I-10707532,3c174ae9e2505a5f9ddbff9843281845,B,590.0,2016-08-01T13:51:14.709000,False,NaN,0.0,False,0,Withdrawn Application,400.0,1199.45,MO,bvMandatory,3,NaN,579eab11e4b0d0502870ef2f,1
3,LL-I-02272596,9be6f443bb97db7e95fa0c281d34da91,B,360.0,2015-08-06T23:58:08.880000,False,NaN,0.0,False,0,Withdrawn Application,500.0,1074.05,IL,bvMandatory,3,NaN,555b1e95e4b0f6f11b267c18,1
4,LL-I-09542882,63b5494f60b5c19c827c7b068443752c,B,590.0,2016-06-05T22:31:34.304000,False,NaN,0.0,False,0,Rejected,350.0,814.37,NV,bvMandatory,3,NaN,5754a91be4b0c6a2bf424772,1


In [4]:
# retrieves the information about the columns, and see if there are any missing values and inpropriate data types
loan_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577682 entries, 0 to 577681
Data columns (total 19 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   loanId                            577426 non-null  object 
 1   anon_ssn                          577682 non-null  object 
 2   payFrequency                      576409 non-null  object 
 3   apr                               573760 non-null  float64
 4   applicationDate                   577682 non-null  object 
 5   originated                        577682 non-null  bool   
 6   originatedDate                    46044 non-null   object 
 7   nPaidOff                          577658 non-null  float64
 8   approved                          577682 non-null  bool   
 9   isFunded                          577682 non-null  int64  
 10  loanStatus                        577291 non-null  object 
 11  loanAmount                        575432 non-null  f

There are total of 19 columns with 577682 rows in this datasets, consisting of 2 binary variables, 7 numerical variables and 10 categorical data.

**Observation**
- `applicationDate` and `originatedDate` are currently stored as object data types and should be converted to datetime format.
- `nPaidOff` is stored as a float variable but represents discrete count data.
- `leadCost` treated it as a continuous numerical feature for easier data processing 

In [5]:
loan_db.describe(include='all')

,loanId,anon_ssn,payFrequency,apr,applicationDate,originated,originatedDate,nPaidOff,approved,isFunded,loanStatus,loanAmount,originallyScheduledPaymentAmount,state,leadType,leadCost,fpStatus,clarityFraudId,hasCF
count,577426,577682,576409,573760.000000,577682,577682,46044,577658.000000,577682,577682.000000,577291,575432.000000,577682.000000,577550,577682,577682.000000,51723,357693,577682.000000
unique,577426,459393,5,NaN,577624,2,46042,NaN,2,NaN,21,NaN,NaN,44,10,NaN,8,314915,NaN
top,LL-I-07399092,c8bb49de1f8ff99d2ecddfb7037dc66e,B,NaN,2017-01-03T18:05:40.811000,False,2015-02-20T19:40:33.329000,NaN,False,NaN,Withdrawn Application,NaN,NaN,OH,bvMandatory,NaN,Checked,561e95f7e4b0efa8a6cdc975,NaN
freq,1,35,316654,NaN,3,531676,2,NaN,537646,NaN,450984,NaN,NaN,90496,475001,NaN,32978,15,NaN
mean,NaN,NaN,NaN,553.080972,NaN,NaN,NaN,0.037887,NaN,0.067480,NaN,514.245084,1428.897209,NaN,NaN,7.854389,NaN,NaN,0.619187
std,NaN,NaN,NaN,110.046159,NaN,NaN,NaN,0.333366,NaN,0.250852,NaN,320.939929,925.009141,NaN,NaN,12.853451,NaN,NaN,0.485587
min,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000,-816.710000,NaN,NaN,0.000000,NaN,NaN,0.000000
25%,NaN,NaN,NaN,490.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,350.000000,1023.640000,NaN,NaN,3.000000,NaN,NaN,0.000000
50%,NaN,NaN,NaN,590.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,500.000000,1245.250000,NaN,NaN,3.000000,NaN,NaN,1.000000
75%,NaN,NaN,NaN,601.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,500.000000,1615.660000,NaN,NaN,6.000000,NaN,NaN,1.000000


**Observation on the summary statistics**:
1. The `originated` and `approved` columns got the most False value denotes that overwhelming majority of loan applications did not pass the underwriting.
2. The percentile of `nPaidOff` columns reveals that over 75% of applicants have never previously paid off a MoneyLion loan.
3. The `isFunded` columns has mean value of 0.067 meaning only 6.75 of applicantions who is funded, indicates this dataset is imbalanced.
4. The `loanAmount` distribution shows that 50% of applicants requested between $350 and $500, with a median of $500 and a mean of $514. 
5. less than 25% of customer does not underwent clarity check.
6. The `originallyScheduledPaymentAmount` column contain a min value of -816.710000, which could be data anomaly.

**Summary: This is imbalanced dataset where only 6.7% of loan applications were funded.
 

In [6]:
loan_db.shape

(577682, 19)

In [7]:
loan_db.isnull().sum()

loanId                                 256
anon_ssn                                 0
payFrequency                          1273
apr                                   3922
applicationDate                          0
originated                               0
originatedDate                      531638
nPaidOff                                24
approved                                 0
isFunded                                 0
loanStatus                             391
loanAmount                            2250
originallyScheduledPaymentAmount         0
state                                  132
leadType                                 0
leadCost                                 0
fpStatus                            525959
clarityFraudId                      219989
hasCF                                    0
dtype: int64

### **1.1 Feature Concatenates - Merging Dataset with Payment Dataset and Clarity Dataset**
In Payment Dataset, the payment data will not included in model training, as the loan repayment information (payment data) does not exists at the time of prediction, including it would cause **Data Leakage**.
<br><br>
In Clarity Underwriting Dataset, I will only retain `clearFraudScore`, as the remaining clarity variables (fraud indicators, inquiry counts, identity verification) are the underlying inputs used to **calculate** the fraud score itself. Including both the score and its underlying variables would introduce **redundant information**, as `clearFraudScore` already summarises them into a single meaningful value.

#### **Payment Dataset**

In [ ]:
payment_db = pd.read_csv('../assets/payment.csv', keep_default_na=False, na_values=['', 'NA', 'N/A'])
payment_db.head()

In [ ]:
payment_db.info()

In [ ]:
payment_db.describe(include='all')

In [ ]:
payment_db.isnull().sum()

In [ ]:
payment_db['paymentStatus'].value_counts()

In [ ]:
payment_db.groupby(['loanId', 'paymentStatus'])['paymentStatus'].count()

#### **Clarity Underwriting Variables Dataset**

In [ ]:
clarity_underwriting_db = pd.read_csv('../assets/clarity_underwriting_variables.csv')
clarity_underwriting_db.head()

In [ ]:
clarity_underwriting_db.info()

In [ ]:
clarity_underwriting_db.describe(include='all')

In [ ]:
clarity_underwriting_db.shape

In [ ]:
clarity_subset = clarity_underwriting_db[['underwritingid', 'clearfraudscore']]

In [ ]:
full_merged_db = loan_db.merge(clarity_subset, left_on='clarityFraudId', right_on='underwritingid', how='left')
full_merged_db.head()

In [ ]:
full_merged_db.info()

In [ ]:
print(f"shape of the dataset: {full_merged_db.shape[0]}, {full_merged_db.shape[1]}")
display(full_merged_db.isnull().sum())

### **1.2 Data Cleaning**

**<strong>`loanId` Column </strong>** <br><br>
I observed that 'laonId' columns contains of 256 missing value, and I handled it by **removing the rows having missing values**.<br>

In [ ]:
# drop empty loan_id rows as they are not useful for analysis
full_merged_db = full_merged_db.dropna(subset=['loanId'])

In [ ]:

# ── 4. Count of loans per loanStatus ──────────────────────────────────
counts = full_merged_db['loanStatus'].value_counts().reset_index()
counts.columns = ['loanStatus', 'count']

fig = px.bar(
    counts,
    x='loanStatus',
    y='count',
    color='loanStatus',
    text='count',
    title='Number of Loans by Loan Status',
    labels={'loanStatus': 'Loan Status', 'count': 'Count'}
)
fig.update_traces(textposition='outside')
fig.update_layout(
    height=500,
    showlegend=False,
    xaxis_tickangle=-45
)
fig.show()

From the result, we can observed that the dataset is heavily dominated by **Withdrawn Applications** (450,984) and **Rejected** loans (85,070), indicates that the majority of applications never reached funded stage.  

So I decided to **filter the dataset to includes only funded loans** since only these loans have repayment outcomes. Non-funded loans do not have repayment information, so includes it would introduces noise and not help in predicting credit risk.

In [ ]:
# Filter the dataset to include only funded loans
full_merged_db = full_merged_db[full_merged_db['isFunded'] == 1]

In [ ]:
full_merged_db.reset_index(drop=True, inplace=True)

In [ ]:
display(full_merged_db.isnull().sum())
display(full_merged_db.shape)

After removing missing values in the `loanId` column and filtering the dataset to funded loan, missing value still remain in the `nPaidOff`, `fpStatus`, `clarityFraudId`, `underwritingid`, and `clearfraudscore` column, which require further data cleaning in the next step.

##### Before imputing, a simply **exploratory data analysis(EDA)** was conducted to understand the behaviour of the missing values.

In [ ]:
# nPaidOff Columns
full_merged_db['loan_count_per_ssn'] = full_merged_db.groupby('anon_ssn')['loanId'].transform('count')
full_merged_db[full_merged_db['nPaidOff'].isnull()][['loanId', 'anon_ssn', 'loan_count_per_ssn', 'nPaidOff', 'leadType']]

- **`nPaidOff`** <br>
The analysis above shows that the applicants with missing values in 'nPaidOff' having a 'loan_count_per_ssn' of 1, from here we can said they have no previous loan history with MoneyLion loan. This reflects that the missing values are not random, but are associated with first-time borrowers.

In [ ]:
# fpStatus column
null_fpStatus = full_merged_db[full_merged_db['fpStatus'].isnull()][['loanId', 'anon_ssn', 'loanStatus', 'fpStatus']]

result = payment_db[payment_db['installmentIndex'] == 1]\
    .merge(null_fpStatus, on='loanId', how='right')\
        [['loanId', 'anon_ssn', 'loanStatus', 'fpStatus', 'installmentIndex', 'paymentDate', 'paymentStatus']]

display(result)

- **`fpStatus`** <br>
By cross-checking against the first installment (installmentIndex = 1) in payment.csv, two distinct cases are observed:<br>
    - **8 loans** (`loanStatus = External Collection`) have their first installment recorded in `payment.csv` with `paymentStatus = 'None'`, indicating that first payment is scheduled but No ACH attempt has been made yet.<br>
    - **1 loan** (`loanStatus = New Loan`) has no record in payment.csv and it is a new loan, indicating it was too new to have any payment schedule set up yet.

In [ ]:
def cramers_v(x, y):
    """
        Cramer's V is used to measure the correlation between two categorical variables.
    """
    confusion_matrix = pd.crosstab(x, y)
    chi2             = chi2_contingency(confusion_matrix)[0]
    n                = confusion_matrix.sum().sum()
    min_dim          = min(confusion_matrix.shape) - 1
    return np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

def correlation_ratio(categories, values):
    """η² (eta squared) — correlation between categorical and numeric"""
    categories  = pd.Series(categories)
    values      = pd.Series(values).dropna()
    categories  = categories[values.index]

    overall_mean = values.mean()
    groups       = categories.unique()

    ss_between = sum(
        len(values[categories == g]) * 
        (values[categories == g].mean() - overall_mean) ** 2
        for g in groups
    )
    ss_total = sum((values - overall_mean) ** 2)

    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0

def compute_mixed_correlation(df, num_cols, cat_cols, bin_cols):
    """
    Compute correlation matrix for mixed numeric and categorical columns
    
    Numeric  vs Numeric     → Pearson
    Numeric  vs Categorical → η² (correlation ratio)
    Categorical vs Categorical → Cramér's V
    """
    all_cols = num_cols + cat_cols + bin_cols
    corr_matrix = pd.DataFrame(
        np.zeros((len(all_cols), len(all_cols))),
        index=all_cols, columns=all_cols
    )

    for col1 in all_cols:
        for col2 in all_cols:
            if col1 == col2:
                corr_matrix.loc[col1, col2] = 1.0

            # Numeric vs Binary 
            elif (col1 in num_cols and col2 in bin_cols) or (col1 in bin_cols and col2 in num_cols):
                bin_col = col1 if col1 in bin_cols else col2
                num_col = col1 if col1 in num_cols else col2
                valid   = df[[num_col, bin_col]].dropna()
                r, _    = pointbiserialr(valid[num_col], valid[bin_col])
                corr_matrix.loc[col1, col2] = r
            
            # Binary vs Binary using Matthews Correlation Coefficient
            elif col1 in bin_cols and col2 in bin_cols:
                valid = df[[col1, col2]].dropna()

                corr_matrix.loc[col1, col2] = matthews_corrcoef(valid[col1], valid[col2])

            # Numeric vs Numeric using Pearson
            elif col1 in num_cols and col2 in num_cols:
                valid = df[[col1, col2]].dropna()
                corr_matrix.loc[col1, col2] = valid[col1].corr(valid[col2])

            # Numeric vs Categorical
            elif (col1 in num_cols and col2 in cat_cols) or (col1 in cat_cols and col2 in num_cols):
                num_col = col1 if col1 in num_cols else col2
                cat_col = col1 if col1 in cat_cols else col2
                valid   = df[[num_col, cat_col]].dropna()
                corr_matrix.loc[col1, col2] = correlation_ratio(
                    valid[cat_col], valid[num_col]
                )

            # Categorical vs Categorical
            else:
                valid = df[[col1, col2]].dropna()
                corr_matrix.loc[col1, col2] = cramers_v(
                    valid[col1], valid[col2]
                )

    return corr_matrix.astype(float).round(2)


# ── Define columns ─────────────────────────────────────────────────────
# excludes binary columns
num_cols = [
    'apr', 'nPaidOff', 'loanAmount',
    'originallyScheduledPaymentAmount', 
    'leadCost', 'clearfraudscore'
]
cat_cols = ['loanStatus', 'payFrequency', 'state', 'leadType', 'fpStatus']
bin_cols = []

# ── Compute mixed correlation matrix ───────────────────────────────────
corr_matrix = compute_mixed_correlation(full_merged_db, num_cols, cat_cols, bin_cols)

# ── Plot heatmap ───────────────────────────────────────────────────────
fig = px.imshow(
    corr_matrix,
    title='Correlation Heatmap',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto=True,
    aspect='auto'
)
fig.update_layout(
    height=600,
    title_font_size=14,
    coloraxis_colorbar=dict(
        title='Correlation',
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['-1\n(negative)', '-0.5', '0\n(none)', '0.5', '1\n(positive)']
    )
)
fig.show()

print("\nCorrelation with clear fraud score:")
print(corr_matrix['clearfraudscore'].sort_values(ascending=False))

**<strong>Observation between `clearfraudscore` and the attributes: </strong>** <br>
From the heatmap, I observed that clearfraudscore has the strongest relationship with `leadType`, `loanAmount`, `originallyScheduledPaymentAmount` and `state`. Thus, I keep these features to support the imputation of missing values in the `clearfraudscore` column. <br>
On the other hand, `leadCost`, `loanStatus`, `payFrequency`, `fpStatus`, `nPaidOff` and `apr` column were excluded due to their weaker relationship with `clearfraudscore` column. <br>
Additional, I included hasCF as well because the missingness of clear fraud score is closely related to whether the underwriting check was performed.

##### **a. Handle Missing Value**

- **`nPaidOff`** <br>
Since the analysis above reflects that the missing values are not random, but are associated with first-time borrowers. <br>
Therefore, I will impute the missing values in ***`nPaidOff = 0`***, representing new laons. <br>

- **`fpStatus`** <br>
Since analysis above shows that the first payment has not happened in both cases, so I will filling missing ***`fpStatus = None`***. 

- **`clarityFraudId`** <br>
For **`clarityFraudId`** columns, I chose not to drop the column because this column will not be used in the model. Instead, I will use the hasCF column, which already indicates whether an underwriting check is present or not. <br>
**<span style="color:yellow">`clarityFraudId` Column will be removed after merging with clarity dataset</span>**

##### **b. Convert Data Types** <br>
- convert `applicationDate` and `originatedDate` to datetime 

In [ ]:

def clean_loan_db(df):
    """
    Cleans the loan database by handling missing values, converting data types and filtering dataset to includes only funded loans.

    Args:
        df (pd.DataFrame): The original loan database to be cleaned.

    Returns:
        pd.DataFrame: A cleaned version of the loan database.  
    """

    # ---- Data Imputation ----
    # nPaidOff: 0 means new customer / new loans.
    df['nPaidOff'] = df['nPaidOff'].fillna(0).astype(int)

    # fpStatus: fill missing values with 'None' meaning that first payment is scheduled but No ACH attempt has been made yet.
    df['fpStatus'] = df['fpStatus'].fillna('None')

    # clearfraudscore: impute missing values using miceforest with lightGBM
    # shallow copy to avoid affecting the original dataframe when converting categorical columns to 'category' dtype
    clearfraudscore_impute = df[['clearfraudscore', 'leadType', 'loanAmount', 'originallyScheduledPaymentAmount', 'state']].copy().reset_index(drop=True)

    # Convert ONLY categorical columns
    cat_cols = ['leadType', 'state']

    # MICE works better when categorical columns are properly marked
    # So we convert all str-type columns to pandas 'category' dtype
    for col in cat_cols:
        clearfraudscore_impute[col] = clearfraudscore_impute[col].astype('category')
        
    kernel = mf.ImputationKernel(
        data=clearfraudscore_impute,
        num_datasets=1,
        save_all_iterations_data=True,
        random_state=42
    )

    kernel.mice(8)

    completed_data = kernel.complete_data()

    # Good convergence:          Bad convergence:
    # Mean                       Mean
    # ────────────               /\/\/\/\
    display(kernel.plot_mean_convergence())

    # Fill the imputed clearfraudscore back to the original dataset
    df.loc[df['clearfraudscore'].isna(), 'clearfraudscore'] = \
    completed_data.loc[clearfraudscore_impute['clearfraudscore'].isna().values, 'clearfraudscore'].values

    # Check the imputed values against the original values
    completed_data.rename(columns={'clearfraudscore': 'clearfraudscore_imputed'}, inplace=True)
    clearfraudscore_impute['clearfraudscore_imputed'] = completed_data['clearfraudscore_imputed']
    print("original values vs Imputed clearfraudscore values :")
    display(clearfraudscore_impute[clearfraudscore_impute['clearfraudscore'].isna()][['clearfraudscore', 'clearfraudscore_imputed']])

    # Drop columns that are not less important for model training and analysis
    df.drop(columns=['loanId', 'anon_ssn', 'originated', 'approved', 'isFunded', \
                     'clarityFraudId', 'underwritingid'], inplace=True)
    
    # ---- date columns ----
    df['applicationDate'] = pd.to_datetime(df['applicationDate'], format='mixed', utc=True, errors='coerce')
    df['originatedDate']  = pd.to_datetime(df['originatedDate'], format='mixed', utc=True, errors='coerce')

    # convert to correct datatypes
    # ---- nPaidOff column ----
    df['nPaidOff'] = pd.to_numeric(df['nPaidOff'], errors='coerce')
    # ---- leadCost column ----
    df['leadCost'] = df['leadCost'].astype(float)

    # ---- categorical standardisation ----
    df['loanStatus'] = df['loanStatus'].str.strip().str.lower()
    df['fpStatus']   = df['fpStatus'].str.strip().str.lower()
    df['leadType']   = df['leadType'].str.strip().str.lower()

    return df

In [ ]:
original_loan_db = full_merged_db.copy()
cleaned_df = full_merged_db.copy()

In [ ]:
cleaned_df = clean_loan_db(cleaned_df)

In [ ]:
cleaned_df.info()

In [ ]:
# check if there are any missing values after the data cleaning process 
print("Loan dataset summary after data cleaning: ")
print(f"Loan dataset shape: {cleaned_df.shape[0]}, {cleaned_df.shape[1]}")
display(cleaned_df.isnull().sum())
print()
display(cleaned_df.head())

In [ ]:
cleaned_df.isnull().sum()

### **1.3 Declare Target Variable**

The target variable is defined based on `loanStatus`, where **1 = Defaulter** and **0 = Non-Defaulter**.

**Defaulter (1):**
- `Charged Off` — customer stopped paying, MoneyLion gave up collecting
- `External Collection` — loan handed to external debt collector
- `Internal Collection` — MoneyLion chasing payment internally
- `Settled Bankruptcy` — customer declared bankruptcy
- `Charged Off Paid Off` — although the customer eventually repaid, they had previously defaulted. <br>
In a real-world scenario, a customer may repeat this pattern multiple times, posing a risk to the business. These cases are flagged as Defaulter and can be escalated to manual review if needed.

**Non-Defaulter (0):**
- `Paid Off Loan` — customer fully repaid the loan
- `Settlement Paid Off` — customer reached an agreement and paid off

**Excluded:**
- `New Loan`, `Returned Item`, `Pending Paid Off`, `Settlement Pending Paid Off` — outcome not yet determined, excluded to avoid ambiguous labels

---

In [ ]:
cleaned_df['loanStatus'].value_counts()

In [ ]:
defaulter = ['charged Off', 'external collection', 'internal collection', 'settled bankruptcy', 'charged off paid off']
non_defaulter = ['paid off loan', 'settlement paid off']
cleaned_df['target'] = cleaned_df['loanStatus'].apply(lambda x: 1 if x in defaulter else (0 if x in non_defaulter else np.nan))

In [ ]:
# excludes ambiguous loans that we cannot confidently label as defaulter or non-defaulter
cleaned_df = cleaned_df.dropna(subset=['target'])
cleaned_df['target'] = cleaned_df['target'].astype(int)

print(f"Remaining rows: {len(cleaned_df)}")
print()
print(cleaned_df.isnull().sum())
print()
print(f"data types of target column is: {cleaned_df['target'].dtypes}")

### **1.4 Data Splitting into Vary Data Types (Discrete, Continuous, Category)**


In [ ]:

def analyze_feature_type(df, threshold=0.05):
    """
    Analyze features in a dataframe to determine if they are likely discrete or continuous.
    
    :param df: Pandas DataFrame to analyze.
    :param threshold: Threshold for determining if a feature is discrete based on the ratio of unique values to total values.
    :return: A dictionary with feature names as keys and their likely type ('discrete' or 'continuous') as values.
    """
    feature_types = {}
    for column in df.columns:
        # Check the data type
        if df[column].dtype == 'int64':
            feature_types[column] = 'discrete'
        elif df[column].dtype == 'float64':
                feature_types[column] = 'continuous'
        else:
            feature_types[column] = 'categorical'
    
    return feature_types

In [ ]:
# Analyze the dataset
feature_column_list = cleaned_df.columns.tolist()
feature_analysis = analyze_feature_type(cleaned_df[feature_column_list])
discrete_features = [feature for feature, feature_type in feature_analysis.items() if feature_type == 'discrete'][:-1]
continuous_features = [feature for feature, feature_type in feature_analysis.items() if feature_type == 'continuous']
cat_features = [feature for feature, feature_type in feature_analysis.items() if feature_type == 'categorical']

In [ ]:
display(discrete_features)
display(continuous_features)
display(cat_features)

In [ ]:
eda_df = cleaned_df.copy()
# save them into dataframe and csv files with target column
df_discrete_features = eda_df[discrete_features]
df_continuous_features = eda_df[continuous_features]
df_categorical_features = eda_df[cat_features]

## **<i><strong>2. EDA - Exploratory Data Analysis</i></strong>**
- Conduct EDA Before Feature Selection
- Plot the distribution of each feature
- Identify Potential Biases or Imbalances in the Dataset

#### **2.1 Plot the distribution of Numeric features and target Variables**

In [ ]:

# Count target values
target_counts = (
    cleaned_df['target']
    .value_counts()
    .reset_index()
)

target_counts.columns = ['target', 'count']

# Optional: rename labels for better readability
target_counts['target'] = target_counts['target'].map({
    0: 'Non-Defaulter',
    1: 'Defaulter'
})

# Pie chart
fig = px.pie(
    target_counts,
    names='target',
    values='count',
    title='Distribution of Target Variable',
    hole=0.3   # optional: donut chart effect
)

fig.update_traces(
    textinfo='percent+label',
    pull=[0, 0.05]   # optional: slightly highlight one slice
)

fig.show()

-----
From the PieChart, the dataset contains a higher proportion of defaulters (58.9%) compared to non-defaulter (41.1%), indicating slight class imbalance. <br> Although the imbalance is not severe and does not require advanced resampling techniques, <br>
it is still important to consider during modeling, as the model may become slightly biased toward predicting the majority class (defaulters).

#### **2.2 Univariate Analysis**

In [ ]:
# ── Auto calculate grid size ───────────────────────────────────────────
n_cols   = 2
n_rows   = math.ceil(len(continuous_features) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes      = axes.flatten()  # flatten to 1D array for easy indexing

# ── Plot each feature ──────────────────────────────────────────────────
for i, feature in enumerate(continuous_features):
    sns.histplot(
        cleaned_df[feature].dropna(),
        kde   = True,
        ax    = axes[i],
        color = '#378ADD'
    )
    axes[i].set_title(
        f'{feature}\nskew={cleaned_df[feature].skew():.2f}',
        fontsize=11
    )
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of Continuous Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

For the continuous numerical features, `clearfraudscore` shows a nearly normal distribution, while `loanAmount`, `originallyScheduledPaymentAmount`, `leadCost` and  are highly right-skewed, indicating that most values are concentrated at lower ranges with a few extreme high values. In contrast, `apr` is moderate left skew, indicating that most loans are associated with higher interest rates.

Summary on this:
- `apr` has left skewed with peak around 600.
- `loanAmount` are right-skewed with a wider spread, indicating greater variability. The long right tail suggests the presences of a small number of large loans, which may represent outliers and require further investigation.
- `originallyScheduledPaymentAmount` mirrors `loanAmount`, showing strongly right skweness with peak between 0-2500
- `leadCost` is highly right-skewed data with multiple peaks suggest that the data was come from different group or segments.
- `clearfraudscore` is nearly normal distribution, with most score concentrated around 700-800. The smooth distribution also indicates that the MICE imputation successfully preserved the natural score distribution.   

In [ ]:
# plt the discrete featrues histogram
cleaned_df[discrete_features].hist(figsize=(20, 20))
plt.show()

The distribution of `nPaifOff` and `loan_count_per_ssn` indicating most of the clients have low activity (few loans and repayments history), resulting in highly right-skewed patterns. `hasCF` column shows that most of the clients has undergoes underwriting check. 

#### **2.2 Bivariate Analysis**

In [ ]:
# Perform bivariate analysis for continuous features against the target variable using boxplots0
features = continuous_features + discrete_features
features.remove('hasCF')  # exclude hasCF as it is binary

n_cols = 2
n_rows = math.ceil(len(features) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.boxplot(
        data=cleaned_df,
        x='target',
        y=feature,
        ax=axes[i], 
        palette='pastel'
    )
    axes[i].set_title(f'{feature} by Target', fontsize=12)
    axes[i].set_xlabel('Target (0 = Non-Defaulter, 1 = Defaulter)')

# remove empty plots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Outliers are observed across all variables, especially in `loanAmount, leadCost, and originallyScheduledPaymentAmount`. <br>
Rather than dealing with outliers, transformation techniques will be used to reduce skewness and the effect of outliers.

In [ ]:
# Generate pairwise plots to visualize the relationships between key numerical variables
g = sns.pairplot(cleaned_df[continuous_features + discrete_features + ['target']], hue='target', palette='pastel', diag_kind='kde')
g.fig.suptitle('Pairwise Relationships of Numerical Variables by Loan Status', y=1.02)

# Loop through the axes to set the ticklabel format
for ax in g.axes.flatten():
    ax.ticklabel_format(style='plain', axis='both')

plt.show()

In [ ]:
# ── Define columns ─────────────────────────────────────────────────────
num_cols = [
    'apr', 'nPaidOff', 'loanAmount',
    'originallyScheduledPaymentAmount', 
    'leadCost', 'clearfraudscore'
]
cat_cols = ['loanStatus', 'payFrequency', 'state', 'leadType', 'fpStatus']
bin_cols = ['hasCF', 'target']

# ── Compute mixed correlation matrix ───────────────────────────────────
corr_matrix = compute_mixed_correlation(cleaned_df, num_cols, cat_cols, bin_cols)

# ── Plot heatmap ───────────────────────────────────────────────────────
fig = px.imshow(
    corr_matrix,
    title='Correlation Heatmap',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto=True,
    aspect='auto'
)
fig.update_layout(
    height=600,
    title_font_size=14,
    coloraxis_colorbar=dict(
        title='Correlation',
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['-1\n(negative)', '-0.5', '0\n(none)', '0.5', '1\n(positive)']
    )
)
fig.show()

print("\nCorrelation with targer variable:")
print(corr_matrix['target'].sort_values(ascending=False))

**Very Strong Positive Correlation** <br>
- <strong>Observation 1: `loanAmount` & `originallyScheduledPaymentAmount` = 0.95</strong>
    - there is highly correlation between `loanAmount` and `originallyScheduledPaymentAmount`.
    - One of it should be removed from model training to avoid data leakage. 

- <strong>Observation 2: `apr` & `state` = 0.8</strong> <br>
    - A correlation of 0.8 suggests that APR is strongly influenced by the state of client
---------
**Strong Positive Correlation** <br>

- <strong>Observation 3: `leadCost` & `leadType` = 0.71</strong> <br>
    - The moderate relationship between these two features is expected since `leadCost` was define by `leadType`.

- <strong>Observation 4: `nPaidOff` & `leadType` = 0.65</strong> <br>
    - This indicates that customers	who	have at	least 1	paid off loan tend to come through specific 
    lead types such as `rc_returning` and `repeat`.
---------
**Moderate Positive Correlation** <br>

- <strong>Observation 5: `loanAmount` & `state` = 0.56</strong> <br>

- <strong>Observation 6: `loanAmount` & `leadType` = 0.54</strong> <br>

- <strong>Observation 7: `state` & `originallyScheduledPaymentAmount` = 0.54</strong> <br>

- <strong>Observation 8: `leadType` & `originallyScheduledPaymentAmount` = 0.51</strong> <br>
---------
**Correlation With target** <br>
- **`loanStatus` = 1.00** — Perfect correlation as expected, since `target` was 
directly derived from `loanStatus`. This confirms that `loanStatus` **should be excluded** from model training to prevent data leakage.

#### **2.3 Multivariate Analysis**

In [ ]:
summary = (
    cleaned_df
    .groupby(['state', 'target'])['apr']
    .mean()
    .reset_index()
)

summary['target_label'] = summary['target'].map({
    0: 'Non-Defaulter',
    1: 'Defaulter'
})

fig = px.bar(
    summary,
    x='state',
    y='apr',
    color='target_label',
    barmode='group',
    title='Average APR by State and Target',
    color_discrete_map = {
    'Non-Defaulter': '#A78BFA',  # soft lavender purple
    'Defaulter': '#86EFAC'       # soft mint green
}
)

fig.update_layout(
    height=500,
    xaxis_title='State',
    yaxis_title='Average APR',
    xaxis_tickangle=45,
    coloraxis_showscale=False
)

fig.show()

In [ ]:
summary = (
    cleaned_df
    .groupby('leadType')['leadCost']
    .mean()
    .reset_index()
)

fig = px.bar(
    summary,
    x='leadType',
    y='leadCost',
    color='leadType',
    title='Average Lead Cost by Lead Type'
)

fig.update_layout(
    showlegend=False,
    height=500
)

fig.show()

In [ ]:
fig = px.strip(
    cleaned_df,
    x='leadType',
    y='nPaidOff',
    color='target',
    title='nPaidOff by Lead Type and Target',
    color_discrete_map={
        0: '#A78BFA',
        1: '#86EFAC'
    }
)

fig.update_layout(height=500)
fig.show()

In [ ]:
px.box(cleaned_df, x='target', y='nPaidOff')

In [ ]:
px.histogram(cleaned_df, x='leadType', color='target')

In [ ]:
for col in cat_cols:
    ct = pd.crosstab(cleaned_df[col], cleaned_df['target'], normalize='index')
    print(f"\n{col} vs Target")
    display(ct)

In [ ]:

for col in num_cols:
    groups = [
        cleaned_df[cleaned_df['target'] == val][col].dropna()
        for val in cleaned_df['target'].unique()
    ]
    f, p = stats.f_oneway(*groups)
    print(f"{col}: p-value = {p:.5f}")

In [ ]:
# # use original loan dataset before any filtering
# eda_df = loan_db.copy()

# pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
# # ── 1. Basic Statistics grouped by loanStatus ─────────────────────────
# print("=== Summary of Loan Amount Statistics by Loan Status ===")
# summary = (
#     eda_df.groupby('loanStatus')
#     .agg(
#         count  = ('loanId',     'count'),
#         sum    = ('loanAmount', 'sum'),
#         mean   = ('loanAmount', 'mean'),
#         median = ('loanAmount', 'median')
#     ).round(2)
# )

# summary['pct_of_total'] = (
#     summary['count'] / summary['count'].sum() * 100
# ).round(2)

# display(summary.sort_values(by='pct_of_total', ascending=False))

# # reset back to default after
# pd.reset_option('display.float_format')

# # # ── 2. Box Plot ────────────────────────────────────────────────────────
# # for col in ['loanAmount', 'apr', 'originallyScheduledPaymentAmount']:
# #     fig = px.box(
# #         eda_df,
# #         x='loanStatus',
# #         y=col,
# #         color='loanStatus',
# #         title=f'Distribution of {col} by Loan Status',
# #         labels={'loanStatus': 'Loan Status', col: col}
# #     )
# #     fig.update_layout(
# #         height=500,
# #         showlegend=False,
# #         xaxis_tickangle=-45
# #     )
# #     fig.show()

# # ── 3. Mean Bar Chart ──────────────────────────────────────────────────
# mean_df = eda_df.groupby('loanStatus')['loanAmount'].mean().reset_index()

# mean_melted = mean_df.melt(
#     id_vars='loanStatus',
#     var_name='feature',
#     value_name='mean_value'
# )

# fig = px.bar(
#     mean_melted,
#     x='loanStatus',
#     y='mean_value',
#     color='loanStatus',
#     facet_col='feature',        # separate chart per feature
#     title='Mean Value of Key Features by Loan Status',
#     labels={
#         'loanStatus' : 'Loan Status',
#         'mean_value' : 'Mean Value',
#         'feature'    : 'Feature'
#     }
# )
# fig.update_layout(
#     height=500,
#     showlegend=False,
#     xaxis_tickangle=-45
# )
# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.show()


## **<i><strong>3. Data-Preprocessing**</i></strong>

In [ ]:
processed_df = cleaned_df.copy()

spr_transformed = np.power(processed_df['apr'], 3)
processed_df['log_apr'] = spr_transformed

# dist plot
plt.figure(figsize = (14,5))
sns.distplot(spr_transformed.dropna())

plt.title(
    f'transformed_apr\nskew={spr_transformed.dropna().skew():.2f}',
    fontsize=12
)
plt.xlabel('')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

# Q-Q plot
plt.figure(figsize=(10,2))
plt.subplot(121)
stats.probplot(cleaned_df['apr'], dist="norm", plot=plt)
plt.title(f'apr Before Cube Transformation')
plt.subplot(122)
stats.probplot(spr_transformed, dist="norm", plot=plt)
plt.title(f'apr After Cube Transformation')
plt.show()

In [ ]:
cols = ['originallyScheduledPaymentAmount', 'loanAmount', 'clearfraudscore']
for col in cols:
    # BoxCox transformation
    boxcox_vals, fitted_lambda = boxcox(cleaned_df[col])
    processed_df[f'log_{col}'] = boxcox_vals

    # convert to 1D array
    log_boxcox_vals = boxcox_vals.flatten()

    # dist plot
    plt.figure(figsize = (14,5))
    sns.distplot(log_boxcox_vals)

    plt.title(
        f'boxcox_{col}\nskew={pd.Series(log_boxcox_vals).skew():.2f}',
        fontsize=12
    )
    plt.xlabel('')
    plt.ylabel('Count')

    plt.tight_layout()
    plt.show()

    # Q-Q plot
    plt.figure(figsize=(10,2))
    plt.subplot(121)
    stats.probplot(cleaned_df[col], dist="norm", plot=plt)
    plt.title(f'{col} Before boxcox')
    plt.subplot(122)
    stats.probplot(log_boxcox_vals, dist="norm", plot=plt)
    plt.title(f'{col} After boxcox')
    plt.show()


#### **3.2 Creating new features**

In [ ]:
processed_df['processing_hours'] = (
    processed_df['originatedDate'] - processed_df['applicationDate']
).dt.total_seconds() / 3600

In [ ]:
processed_df['payment_to_loan_ratio'] = (
    processed_df['originallyScheduledPaymentAmount'] / 
    processed_df['loanAmount'].replace(0, np.nan)  # ← safe division ✅
)

# Check for any infinite or null values
print(processed_df['payment_to_loan_ratio'].notna().sum())  # should be True

In [ ]:
state= pd.crosstab(processed_df["state"], processed_df["target"])
state['default_ratio_per_state'] = (state[1] / (state[0] + state[1])).fillna(0)
state= state.drop([0, 1], axis=1)
state.index.set_names('state', inplace=True)
state.head()

In [ ]:
# inner joining ratio with the 'funded_class' dataset on 'state'
processed_df= pd.merge(processed_df, state, on='state', how='left')
processed_df.isnull().sum()

In [ ]:
processed_df.drop(columns=['loanStatus', 'applicationDate', 'originatedDate', 'state', 'fpStatus', 'originallyScheduledPaymentAmount', 'loanAmount'], inplace=True)

In [ ]:
processed_df.columns.tolist()

#### **3.3 Encoding Categorical Variables**

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

categorical_columns = ['payFrequency', 'leadType']

# Creating the encoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fitting the encoder to the data
one_hot_encoded = encoder.fit_transform(processed_df[categorical_columns])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns), )

df_encoded = pd.concat([processed_df, one_hot_df], axis=1)

df_encoded = df_encoded.drop(categorical_columns, axis=1)
print(f"Encoded Employee data : \n{df_encoded}")

In [ ]:
df_encoded.columns.tolist()

## **<i><strong>4. Model Training**</i></strong>
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- Gradient Boost
- Cat Boost

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##### **4.1 Logistic Regression**
- Statistical Model

In [ ]:
from sklearn.preprocessing import StandardScaler

X_train_lg = X_train.copy()
X_test_lg = X_test.copy()

scaler = StandardScaler()

X_train_lg = scaler.fit_transform(X_train_lg)
X_test_lg = scaler.transform(X_test_lg)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = [
    (
        'Logistic Regression', 
        LogisticRegression(solver='lbfgs', max_iter=200, random_state=42),
        (X_train_lg, y_train),
        (X_test_lg, y_test)
    ),

    (
        'Decision Tree',
        DecisionTreeClassifier(random_state=42),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        'Random Forest',
        RandomForestClassifier(n_estimators=100, random_state=42),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        'XGBoost',
        XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        'Gradient Boosting',
        GradientBoostingClassifier(random_state=42),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        'CatBoost',
        CatBoostClassifier(verbose=0, random_state=42),
        (X_train, y_train),
        (X_test, y_test)
    )
]

In [ ]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, auc
from sklearn.model_selection import learning_curve

reports = []
plt.figure(figsize=(10,6))

for model_name, model, train_set, test_set in models:
    X_train, y_train = train_set
    X_test, y_test = test_set

    model.fit(X_train, y_train)

    # print(f"\n=== {model_name} ===")
    # ======================
    # TRAINING PREDICTION
    # ======================

    y_train_pred = model.predict(X_train)

    train_accuracy = accuracy_score(
        y_train, 
        y_train_pred
    )

    training_report = classification_report(
        y_train, 
        y_train_pred, 
        output_dict=True
    )

    # print(f"Training Accuracy: \n {classification_report(y_train, y_train_pred)}")
    # ======================
    # TEST PREDICTION
    # ======================

    y_probs = model.predict_proba(X_test)[:, 1]

    y_pred = model.predict(X_test)

    test_accuracy = accuracy_score(
        y_test, 
        y_pred
    )

    testing_report = classification_report(
        y_test, 
        y_pred, 
        output_dict=True
    )
    
    # print(f"Testing Accuracy: \n {classification_report(y_test, y_pred)}")

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5)
    )

    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)

    plt.plot(
        train_sizes,
        val_mean,
        label=f"{model_name}"
    )

    reports.append((model_name, 
                    round(train_accuracy, 2), 
                    round(test_accuracy, 2), testing_report, y_test, y_probs))
    
plt.xlabel("Training Size")
plt.ylabel("F1 Macro Score")

plt.title("Validation Learning Curves")

plt.legend()

plt.show()

In [ ]:
for model_name, y_test, y_probs in [(report[0], report[4], report[5]) for report in reports]:
    fpr, tpr, thresholds = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    fig = px.area(
        x=fpr, y=tpr,
        title=f'{model_name} - ROC Curve (AUC={roc_auc:.4f})',
        labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
        width=700, height=500
    )

    # 4. Add diagonal reference line (random guess)
    fig.add_shape(
        type='line', line=dict(dash='dash'),
        x0=0, x1=1, y0=0, y1=1
    )

    fig.show()

In [ ]:
for model_name, model, train_set, test_set in models:

    X_train, y_train = train_set
    X_test, y_test = test_set

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10)
    )

    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)

    plt.figure(figsize=(8,5))

    plt.plot(train_sizes, train_mean, label='Training Score')
    plt.plot(train_sizes, val_mean, label='Validation Score')

    plt.xlabel("Training Size")
    plt.ylabel("F1 Macro Score")
    plt.title(f"{model_name} Learning Curve")
    plt.legend()

    plt.show()

In [ ]:
from sklearn.calibration import calibration_curve

for model_name, y_test,y_probs in [(report[0], report[3], report[4]) for report in reports]:
 
    # Compute calibration curve
    prob_true, prob_pred = calibration_curve(y_test, y_probs, n_bins=10)

    # Plot the calibration curve
    plt.figure(figsize=(8, 6))
    plt.plot(prob_pred, prob_true, marker='o', label='Calibration Curve', color='blue')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Ideal Calibration')
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives')
    plt.title(f'{model_name} - Calibration Curve')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# MLflow
# Tracking experiment metadata
mlflow.set_experiment("ML Experiment")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(model.get_params())
        # Log experiment metadata
        mlflow.log_param('accuracy', report[1])
        mlflow.log_param('recall_class_1', report[2]['1']['recall'])
        mlflow.log_param('recall_class_0', report[2]['0']['recall'])
        mlflow.log_param('precision_class_1', report[2]['1']['precision'])
        mlflow.log_param('precision_class_0', report[2]['0']['precision'])
        mlflow.log_param('f1_score_macro', report[2]['macro avg']['f1-score'])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

## **<i><strong>5. Model Tuning**</i></strong>
- Grid Search
- Random Search

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Define the parameter grid to tune the hyperparameters
# Decision Tree
param_grid_dt = {
        'criterion': ['gini', 'entropy', None],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

# Random Forest
param_grid_rf = { 
    'max_depth': [5,10,20],
    'min_samples_leaf': [1,2],
    'n_estimators': [100,200],
    'max_features': ['sqrt']
}

In [ ]:
decision_tree, dt_model, train_set_dt, test_set_dt = models[1]
X_train_dt, y_train_dt = train_set_dt
X_test_dt, y_test_dt = test_set_dt

grid_search_dt = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid_dt,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='accuracy'
    )

grid_search_dt.fit(X_train_dt, y_train_dt)
best_model_dt = grid_search_dt.best_estimator_ # Get the best estimator from the grid search

# ======================
# Training PREDICTION
# ======================
xtrain_pred_dt = best_model_dt.predict(X_train_dt)

training_report_dt = classification_report(y_train_dt, xtrain_pred_dt)
print(f"Training Accuracy: \n {training_report_dt}")

# ======================
# Teat PREDICTION
# ======================
y_probs_dt = best_model_dt.predict_proba(X_test_dt)[:, 1]
y_pred_dt = best_model_dt.predict(X_test_dt)
 
testing_report_dt = classification_report(y_test_dt, y_pred_dt)
print(f"Testing Accuracy: \n {testing_report_dt}")

print(f"\nModel: {decision_tree}")
print(f"Best parameters: {grid_search_dt.best_params_}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_dt, y_probs_dt)
roc_auc = auc(fpr, tpr)

fig = px.area(
    x=fpr, y=tpr,
    title=f'{decision_tree} - ROC Curve (AUC={roc_auc:.4f})',
    labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
    width=700, height=500
)

# 4. Add diagonal reference line (random guess)
fig.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)

fig.show()

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_model_dt,
    X=X_train_dt,
    y=y_train_dt,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(8,5))

plt.plot(train_sizes, train_mean, label='Training Score')
plt.plot(train_sizes, val_mean, label='Validation Score')

plt.xlabel("Training Size")
plt.ylabel("F1 Macro Score")
plt.title(f"{decision_tree} Learning Curve")
plt.legend()

plt.show()

In [ ]:
random_forest, rf_model, train_set_rf, test_set_rf = models[2]
X_train_rf, y_train_rf = train_set_rf
X_test_rf, y_test_rf = test_set_rf

grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='f1_macro'
    )

grid_search_rf.fit(X_train_rf, y_train_rf)
best_model_rf = grid_search_rf.best_estimator_ # Get the best estimator from the grid search

# ======================
# Training PREDICTION
# ======================
xtrain_pred_rf = best_model_rf.predict(X_train_rf)

training_report_rf = classification_report(y_train_rf, xtrain_pred_rf)
print(f"Training Accuracy: \n {training_report_rf}")

# ======================
# Testing PREDICTION
# ======================
y_probs_rf = best_model_rf.predict_proba(X_test_rf)[:, 1]
y_pred_rf = best_model_rf.predict(X_test_rf)

testing_report_rf = classification_report(y_test_rf, y_pred_rf)
print(f"Testing Accuracy: \n {testing_report_rf}")

print(f"\nModel: {random_forest}")
print(f"Best parameters: {grid_search_rf.best_params_}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_rf, y_probs_rf)
roc_auc = auc(fpr, tpr)

fig = px.area(
    x=fpr, y=tpr,
    title=f'{random_forest} - ROC Curve (AUC={roc_auc:.4f})',
    labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
    width=700, height=500
)

# 4. Add diagonal reference line (random guess)
fig.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)

fig.show()

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_model_rf,
    X=X_train_rf,
    y=y_train_rf,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(8,5))

plt.plot(train_sizes, train_mean, label='Training Score')
plt.plot(train_sizes, val_mean, label='Validation Score')

plt.xlabel("Training Size")
plt.ylabel("F1 Macro Score")
plt.title(f"{random_forest} Learning Curve")
plt.legend()

plt.show()

In [ ]:
model = models[0][1] 
# Coefficients and Odds Ratios
coefficients = model.coef_[0]
odds_ratios = np.exp(coefficients)


# Display feature importance using coefficients and odds ratios
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients,
    'Odds Ratio': odds_ratios
})
print("\nFeature Importance (Coefficient and Odds Ratio):")
print(feature_importance.sort_values(by='Coefficient', ascending=False))

In [ ]:
import xgboost as xgb

model = models[3][1] 

plt.figure(figsize=(8,6))
xgb.plot_importance(model)
plt.title("Feature Importance")
plt.show()